# CareLoop Clinics – Part 2: SQL Verification & No-Show Clustering
**Tasks 2.1 – 2.5: Build Database, Verify Counts, and Answer Business Questions with SQL**

This notebook executes the SQL scripts in `sql/` against `sql/load_data.sql` using SQLite (MySQL 8–compatible queries) and cross-validates clinic aggregates against Pandas `groupby`.

## Task 2.1 – Build and Load the Database (4 Marks)
Create `clinics` and `appointments` tables from `sql/create_tables.sql`, execute `sql/load_data.sql`, verify `COUNT(*)` and `SUM(no_show)`, and test the duplicate primary key constraint on `'A0001'`.

In [1]:
import os
import sqlite3
import pandas as pd

PROJECT_ROOT = ".." if os.path.basename(os.getcwd()) == "notebooks" else "."
SQL_DIR = os.path.join(PROJECT_ROOT, "sql")
PROCESSED_DIR = os.path.join(PROJECT_ROOT, "data", "processed")
OUTPUTS_DIR = os.path.join(PROJECT_ROOT, "outputs")
os.makedirs(OUTPUTS_DIR, exist_ok=True)

db_path = os.path.join(OUTPUTS_DIR, "noshowguard.db")
if os.path.exists(db_path):
    os.remove(db_path)

conn = sqlite3.connect(db_path)
conn.execute("PRAGMA foreign_keys = ON;")

with open(os.path.join(SQL_DIR, "create_tables.sql"), "r", encoding="utf-8") as f:
    conn.executescript(f.read())

with open(os.path.join(SQL_DIR, "load_data.sql"), "r", encoding="utf-8") as f:
    conn.executescript(f.read())

clinic_cnt = pd.read_sql_query("SELECT COUNT(*) AS clinic_count FROM clinics;", conn)
appt_cnt = pd.read_sql_query("SELECT COUNT(*) AS appointment_count FROM appointments;", conn)
noshow_sum = pd.read_sql_query("SELECT SUM(no_show) AS total_no_shows FROM appointments;", conn)

print("Clinics loaded     :", int(clinic_cnt["clinic_count"].iloc[0]))
print("Appointments loaded:", int(appt_cnt["appointment_count"].iloc[0]))
print("Total no-shows     :", int(noshow_sum["total_no_shows"].iloc[0]))

# Duplicate primary key test on 'A0001'
try:
    conn.execute(
        """INSERT INTO appointments VALUES (
            'A0001','P509','C05','2025-01-24','2025-01-25',60,'F',18.0,850,
            'Phone','Consultation','Yes',1,'No-Show',1,1
        );"""
    )
    conn.commit()
except sqlite3.IntegrityError as exc:
    print("Duplicate insert error recorded:", repr(exc))

Clinics loaded     : 8
Appointments loaded: 990
Total no-shows     : 218
Duplicate insert error recorded: IntegrityError('UNIQUE constraint failed: appointments.appointment_id')


**Interpretation (Task 2.1):** All 8 clinics, 990 appointments, and 218 no-shows match Part 1 exactly. Inserting a second row with `appointment_id = 'A0001'` fails with a primary-key `UNIQUE constraint failed` error (in MySQL 8: `Duplicate entry 'A0001' for key 'appointments.PRIMARY'`), confirming that the primary key prevents duplicate appointment IDs.

## Task 2.2 – No-Show Rate by Clinic (5 Marks)
Run `sql/task_2_2_clinic_analysis.sql` (with `HAVING COUNT(*) >= 95`, then all 8 clinics without `HAVING`) and cross-validate against Pandas `groupby`.

In [2]:
with open(os.path.join(SQL_DIR, "task_2_2_clinic_analysis.sql"), "r", encoding="utf-8") as f:
    sql_2_2 = f.read()

queries_2_2 = [q.strip() for q in sql_2_2.split(";") if "SELECT" in q.upper()]
df_having = pd.read_sql_query(queries_2_2[0], conn)
df_all_sql = pd.read_sql_query(queries_2_2[1], conn)

print("Clinics with >= 95 appointments:")
print(df_having.to_string(index=False))

print("\nAll 8 clinics (without HAVING):")
print(df_all_sql.to_string(index=False))

# Python Pandas cross-validation
appts_df = pd.read_csv(os.path.join(PROCESSED_DIR, "appointments_clean.csv"))
clinics_df = pd.read_csv(os.path.join(PROCESSED_DIR, "clinics_enriched.csv"))

df_pd = (
    appts_df.merge(clinics_df[["clinic_id", "clinic_name", "city"]], on="clinic_id", how="left")
    .groupby(["clinic_name", "city"], as_index=False)
    .agg(appointments=("appointment_id", "count"), no_shows=("no_show", "sum"))
)
df_pd["no_show_pct"] = (100.0 * df_pd["no_shows"] / df_pd["appointments"]).round(1)
df_pd = df_pd.sort_values("no_show_pct", ascending=False).reset_index(drop=True)

pd.testing.assert_frame_equal(df_all_sql, df_pd)
print("\nPandas groupby matches SQL results exactly: True")

Clinics with >= 95 appointments:
             clinic_name   city  appointments  no_shows  no_show_pct
    Hadapsar Care Centre   Pune           143        39         27.3
    Malviya Nagar Clinic Jaipur           142        38         26.8
   Kothrud Family Clinic   Pune           160        36         22.5
      Vijay Nagar Clinic Indore           101        21         20.8
Edappally Medical Centre  Kochi           142        23         16.2
   Vaishali Nagar Clinic Jaipur           130        18         13.8

All 8 clinics (without HAVING):
             clinic_name    city  appointments  no_shows  no_show_pct
  Gomti Nagar Care Point Lucknow            90        30         33.3
    Hadapsar Care Centre    Pune           143        39         27.3
    Malviya Nagar Clinic  Jaipur           142        38         26.8
   Kothrud Family Clinic    Pune           160        36         22.5
      Vijay Nagar Clinic  Indore           101        21         20.8
Edappally Medical Centre   Koch

**Interpretation (Task 2.2):** Among clinics with ≥ 95 appointments, Hadapsar Care Centre (Pune, 27.3%) and Malviya Nagar Clinic (Jaipur, 26.8%) have the highest no-show rates. Removing `HAVING` reveals that Gomti Nagar Care Point (Lucknow) actually has the highest no-show rate overall (**33.3%**, 90 appointments), while Sitabuldi Health Point (Nagpur) has 82 appointments at 15.9%. Threshold filtering hides smaller-volume clinics with severe operational issues.

## Task 2.3 – Does Booking Far Ahead Matter? (4 Marks)
Execute `sql/task_2_3_lead_time.sql` using `CASE WHEN` bands (`1-7`, `8-14`, `15-30`, `31+`) ordered chronologically by `MIN(lead_days)`.

In [3]:
with open(os.path.join(SQL_DIR, "task_2_3_lead_time.sql"), "r", encoding="utf-8") as f:
    sql_2_3 = f.read()

df_2_3 = pd.read_sql_query(sql_2_3, conn)
print(df_2_3.to_string(index=False))

lead_band  appointments  no_shows  no_show_pct
      1-7           390        63         16.2
     8-14           304        53         17.4
    15-30           253        76         30.0
      31+            43        26         60.5


**Interpretation (Task 2.3):** No-show rates rise monotonically with booking lead time: `1-7` days (`16.2%`), `8-14` days (`17.4%`), `15-30` days (`30.0%`), and `31+` days (`60.5%`). Longer lead times are strongly associated with missed appointments (though this observational breakdown does not establish causation).

## Task 2.4 – Month-by-Month Trend (4 Marks)
Execute `sql/task_2_4_monthly_trend.sql` using the `monthly` CTE and `LAG() OVER (ORDER BY month)`.

In [4]:
with open(os.path.join(SQL_DIR, "task_2_4_monthly_trend.sql"), "r", encoding="utf-8") as f:
    sql_2_4 = f.read()

df_2_4 = pd.read_sql_query(sql_2_4, conn)
print(df_2_4.to_string(index=False))

  month  appointments  no_show_pct  change_pp
2025-01           157         24.8        NaN
2025-02           151         20.5       -4.3
2025-03           142         20.4       -0.1
2025-04           177         20.9        0.5
2025-05           198         23.7        2.8
2025-06           165         21.2       -2.5


**Interpretation (Task 2.4):** The highest monthly no-show rate occurred in `2025-01` (`24.8%`), and the lowest in `2025-03` (`20.4%`). The largest month-over-month decrease was `-4.3` percentage points in `2025-02`, and the largest increase was `+2.8` percentage points in `2025-05`. January (`2025-01`) has `NULL` for `change_pp` because no prior month exists.

## Task 2.5 – Rank Clinics Inside Each City (5 Marks)
Execute `sql/task_2_5_city_ranking.sql` using the `clinic_rates` CTE, `RANK() OVER (PARTITION BY city ORDER BY no_show_pct DESC)`, and a scalar subquery for the unrounded overall rate.

In [5]:
overall_rate = pd.read_sql_query(
    "SELECT 100.0 * SUM(no_show) / COUNT(*) AS overall_pct FROM appointments;", conn
)["overall_pct"].iloc[0]
print(f"Unrounded overall no-show rate: {overall_rate:.4f}%")

with open(os.path.join(SQL_DIR, "task_2_5_city_ranking.sql"), "r", encoding="utf-8") as f:
    sql_2_5 = f.read()

df_2_5 = pd.read_sql_query(sql_2_5, conn)
print(df_2_5.to_string(index=False))

conn.close()

Unrounded overall no-show rate: 22.0202%
   city              clinic_name  no_show_pct  rank_in_city vs_overall
 Indore       Vijay Nagar Clinic         20.8             1      below
 Jaipur     Malviya Nagar Clinic         26.8             1      above
 Jaipur    Vaishali Nagar Clinic         13.8             2      below
  Kochi Edappally Medical Centre         16.2             1      below
Lucknow   Gomti Nagar Care Point         33.3             1      above
 Nagpur   Sitabuldi Health Point         15.9             1      below
   Pune     Hadapsar Care Centre         27.3             1      above
   Pune    Kothrud Family Clinic         22.5             2      above


**Interpretation (Task 2.5):** Compared against the unrounded overall rate (`22.02%`), four clinics are `above` overall (Malviya Nagar Clinic in Jaipur, Gomti Nagar Care Point in Lucknow, and both Pune clinics—Hadapsar Care Centre and Kothrud Family Clinic), while four are `below` overall (Vijay Nagar Clinic, Vaishali Nagar Clinic, Edappally Medical Centre, and Sitabuldi Health Point).